# Основы построения нейронных сетей с PyTorch, Keras/TensorFlow и Hugging Face



## Установка зависимостей

Минимальный набор для этого занятия:

    python -m pip install numpy matplotlib torch tensorflow transformers accelerate

Опционально для реальных проектов:

    python -m pip install scikit-learn pandas datasets evaluate tensorboard optuna

Примечания:

- PyTorch и TensorFlow лучше ставить по официальным инструкциям под вашу ОС, GPU/CUDA или Apple Silicon.
- Если TensorFlow не импортируется из-за несовместимой сборки CPU, проще запустить этот notebook в Google Colab или создать новое окружение с совместимой версией Python/TensorFlow.
- Hugging Face pretrained-модели скачиваются при первом запуске. Offline-пример ниже не скачивает веса, но требует установленный пакет transformers.


## Официальные источники API

Этот notebook опирается на текущие официальные гайды:

- PyTorch: https://docs.pytorch.org/tutorials/recipes/recipes/defining_a_neural_network.html
- PyTorch train loop / zero_grad: https://docs.pytorch.org/tutorials/recipes/recipes/zeroing_out_gradients.html
- Keras Sequential model: https://www.tensorflow.org/guide/keras/sequential_model
- TensorFlow/Keras compile-fit workflow: https://www.tensorflow.org/tutorials/images/classification
- Hugging Face Trainer: https://huggingface.co/docs/transformers/main_classes/trainer
- Hugging Face text classification: https://huggingface.co/docs/transformers/main/tasks/sequence_classification


## 1. Общая схема обучения нейросети

Почти любая supervised-задача выглядит так:

1. Данные: входы X и целевые метки y.
2. Модель: функция с обучаемыми параметрами, которая превращает X в прогноз.
3. Loss: число, которое показывает, насколько прогноз плох.
4. Backpropagation: вычисление градиентов loss по параметрам модели.
5. Optimizer: правило обновления параметров по градиентам.
6. Validation: проверка качества на данных, которые модель не видела во время обучения.
7. Inference: применение обученной модели к новым данным.

Главные формы данных:

- Табличные признаки / векторы: batch x features.
- Изображения в PyTorch: batch x channels x height x width.
- Изображения в Keras/TensorFlow: batch x height x width x channels.
- Текст для Transformers: batch x sequence_length токенов + attention_mask.

Словарь основных терминов:

- batch: небольшая порция данных, на которой считается один шаг обновления весов.
- epoch: один полный проход по train dataset.
- logits: сырые выходы модели до softmax/sigmoid.
- loss: функция ошибки, по которой модель обучается.
- metric: показатель качества для человека, например accuracy или F1.
- learning rate: размер шага оптимизатора.
- regularization: приемы против переобучения, например dropout и weight decay.


In [ ]:
import math
import random
from typing import Dict, List, Tuple

import numpy as np
import matplotlib.pyplot as plt

SEED = 42
random.seed(SEED)
np.random.seed(SEED)

plt.rcParams["figure.figsize"] = (6, 4)
plt.rcParams["axes.grid"] = True

print("NumPy:", np.__version__)


## 2. Учебный датасет

Сделаем синтетическую задачу бинарной классификации: две переплетенные дуги. Она удобна для занятия, потому что линейная модель справляется плохо, а небольшая нейросеть с нелинейностями быстро учится.


In [ ]:
def make_two_moons_np(n: int = 1400, noise: float = 0.16, seed: int = 42) -> Tuple[np.ndarray, np.ndarray]:
    rng = np.random.default_rng(seed)
    n_first = n // 2
    n_second = n - n_first

    t0 = rng.uniform(0.0, math.pi, size=n_first)
    x0 = np.column_stack([np.cos(t0), np.sin(t0)])

    t1 = rng.uniform(0.0, math.pi, size=n_second)
    x1 = np.column_stack([1.0 - np.cos(t1), 0.45 - np.sin(t1)])

    X = np.vstack([x0, x1])
    y = np.concatenate([
        np.zeros(n_first, dtype=np.int64),
        np.ones(n_second, dtype=np.int64),
    ])

    X = X + rng.normal(scale=noise, size=X.shape)
    permutation = rng.permutation(n)
    X = X[permutation]
    y = y[permutation]

    X = (X - X.mean(axis=0)) / (X.std(axis=0) + 1e-7)
    return X.astype("float32"), y.astype("int64")


X, y = make_two_moons_np()
split = int(0.8 * len(X))
X_train, X_valid = X[:split], X[split:]
y_train, y_valid = y[:split], y[split:]

print("X_train:", X_train.shape, X_train.dtype)
print("y_train:", y_train.shape, y_train.dtype)
print("class balance:", np.bincount(y_train))

plt.scatter(X[:, 0], X[:, 1], c=y, cmap="coolwarm", s=12, alpha=0.75)
plt.title("Toy binary classification dataset")
plt.xlabel("feature 1")
plt.ylabel("feature 2")
plt.show()


In [ ]:
def plot_binary_decision_boundary(predict_proba_fn, X: np.ndarray, y: np.ndarray, title: str) -> None:
    x_min, x_max = X[:, 0].min() - 0.5, X[:, 0].max() + 0.5
    y_min, y_max = X[:, 1].min() - 0.5, X[:, 1].max() + 0.5
    xs = np.linspace(x_min, x_max, 240)
    ys = np.linspace(y_min, y_max, 240)
    xx, yy = np.meshgrid(xs, ys)
    grid = np.column_stack([xx.ravel(), yy.ravel()]).astype("float32")

    proba = predict_proba_fn(grid).reshape(xx.shape)

    plt.figure(figsize=(5.5, 4.5))
    plt.contourf(xx, yy, proba, levels=24, cmap="RdBu", alpha=0.35)
    plt.scatter(X[:, 0], X[:, 1], c=y, cmap="coolwarm", s=10, alpha=0.75)
    plt.title(title)
    plt.xlabel("feature 1")
    plt.ylabel("feature 2")
    plt.show()


def plot_history_curves(history: Dict[str, List[float]], title: str) -> None:
    fig, axes = plt.subplots(1, 2, figsize=(10, 3.5))

    axes[0].plot(history["train_loss"], label="train")
    axes[0].plot(history["valid_loss"], label="valid")
    axes[0].set_title(f"{title}: loss")
    axes[0].set_xlabel("epoch")
    axes[0].legend()

    axes[1].plot(history["train_acc"], label="train")
    axes[1].plot(history["valid_acc"], label="valid")
    axes[1].set_title(f"{title}: accuracy")
    axes[1].set_xlabel("epoch")
    axes[1].legend()

    plt.tight_layout()
    plt.show()


## 3. PyTorch: модель и ручной цикл обучения

PyTorch дает явный контроль над каждым шагом обучения.

Основные элементы PyTorch pipeline:

1. torch.Tensor: основная структура данных, похожая на NumPy array, но умеет работать на GPU и хранить граф вычислений.
2. Dataset: объект с данными. В простом случае можно использовать TensorDataset.
3. DataLoader: делает батчи, перемешивает train set и удобно итерируется в цикле.
4. nn.Sequential: простой способ собрать модель как последовательность слоев без собственного класса.
5. nn.Linear: полносвязный слой.
6. nn.ReLU: нелинейность, без нее несколько Linear-слоев почти эквивалентны одному Linear-слою.
7. nn.BatchNorm1d: стабилизирует распределение активаций и часто ускоряет обучение.
8. nn.Dropout: случайно отключает часть нейронов на train, снижая переобучение.
9. nn.CrossEntropyLoss: loss для multiclass-классификации по logits и целым номерам классов.
10. torch.optim.AdamW: оптимизатор, который обновляет веса и поддерживает weight decay.
11. scheduler: меняет learning rate во время обучения.
12. model.train() и model.eval(): переключают поведение Dropout/BatchNorm между обучением и оценкой.
13. torch.no_grad(): отключает расчет градиентов при validation/inference.

Pipeline работы:

1. Превратить NumPy arrays в torch.Tensor.
2. Создать TensorDataset и DataLoader.
3. Собрать модель через nn.Sequential.
4. Выбрать loss и optimizer.
5. В цикле обучения выполнить zero_grad -> forward -> loss -> backward -> step.
6. На validation выполнить forward без градиентов и посчитать метрики.
7. Для inference применить softmax/sigmoid к logits, если нужны вероятности.


In [ ]:
try:
    import torch
    from torch import nn
    from torch.utils.data import DataLoader, TensorDataset
except ModuleNotFoundError as exc:
    raise ModuleNotFoundError(
        "PyTorch не установлен. Выполните: python -m pip install torch"
    ) from exc


def get_torch_device() -> torch.device:
    if torch.cuda.is_available():
        return torch.device("cuda")
    return torch.device("cpu")


torch.manual_seed(SEED)
DEVICE = get_torch_device()
print("PyTorch:", torch.__version__)
print("Device:", DEVICE)


In [ ]:
X_train_t = torch.tensor(X_train, dtype=torch.float32)
y_train_t = torch.tensor(y_train, dtype=torch.long)
X_valid_t = torch.tensor(X_valid, dtype=torch.float32)
y_valid_t = torch.tensor(y_valid, dtype=torch.long)

train_dataset = TensorDataset(X_train_t, y_train_t)
valid_dataset = TensorDataset(X_valid_t, y_valid_t)

train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)
valid_loader = DataLoader(valid_dataset, batch_size=128)

batch_x, batch_y = next(iter(train_loader))
print("batch_x:", batch_x.shape)
print("batch_y:", batch_y.shape)


In [ ]:
INPUT_DIM = 2
HIDDEN_DIM = 64
NUM_CLASSES = 2
DROPOUT = 0.10


def build_torch_mlp(
    input_dim: int = 2,
    hidden_dim: int = 64,
    num_classes: int = 2,
    dropout: float = 0.10,
) -> nn.Sequential:
    """Собирает MLP без собственного класса модели."""
    return nn.Sequential(
        nn.Linear(input_dim, hidden_dim),      # признаки -> скрытое представление
        nn.ReLU(),                            # нелинейность
        nn.BatchNorm1d(hidden_dim),           # стабилизация активаций
        nn.Dropout(dropout),                  # регуляризация
        nn.Linear(hidden_dim, hidden_dim),
        nn.ReLU(),
        nn.Linear(hidden_dim, num_classes),   # logits по числу классов
    )


model_torch = build_torch_mlp(
    input_dim=INPUT_DIM,
    hidden_dim=HIDDEN_DIM,
    num_classes=NUM_CLASSES,
    dropout=DROPOUT,
).to(DEVICE)

print(model_torch)

with torch.no_grad():
    logits = model_torch(batch_x.to(DEVICE))
print("logits shape:", logits.shape)
print("first logits:", logits[:3])


In [ ]:
@torch.no_grad()
def evaluate_torch(model: nn.Module, loader: DataLoader, criterion: nn.Module) -> Tuple[float, float]:
    # Validation/inference: градиенты не нужны, поэтому функция обернута в torch.no_grad().
    model.eval()
    total_loss = 0.0
    total_correct = 0
    total_count = 0

    for xb, yb in loader:
        xb = xb.to(DEVICE)
        yb = yb.to(DEVICE)

        # Forward pass: получаем logits.
        logits = model(xb)
        loss = criterion(logits, yb)

        total_loss += loss.item() * yb.size(0)
        total_correct += (logits.argmax(dim=1) == yb).sum().item()
        total_count += yb.size(0)

    return total_loss / total_count, total_correct / total_count


def train_torch_classifier(
    model: nn.Module,
    train_loader: DataLoader,
    valid_loader: DataLoader,
    epochs: int = 35,
    lr: float = 3e-3,
    weight_decay: float = 1e-4,
) -> Dict[str, List[float]]:
    # 1. Loss знает, как сравнить logits модели и правильные классы.
    criterion = nn.CrossEntropyLoss()

    # 2. Optimizer знает, какие параметры обновлять и с каким learning rate.
    optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=weight_decay)

    # 3. Scheduler уменьшает learning rate, если validation loss перестал улучшаться.
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer,
        mode="min",
        factor=0.5,
        patience=4,
    )

    history = {"train_loss": [], "train_acc": [], "valid_loss": [], "valid_acc": []}

    for epoch in range(1, epochs + 1):
        model.train()
        total_loss = 0.0
        total_correct = 0
        total_count = 0

        for xb, yb in train_loader:
            xb = xb.to(DEVICE)
            yb = yb.to(DEVICE)

            # Полный шаг обучения PyTorch:
            optimizer.zero_grad(set_to_none=True)  # очистить старые градиенты
            logits = model(xb)                     # forward pass
            loss = criterion(logits, yb)           # посчитать ошибку
            loss.backward()                        # backpropagation
            optimizer.step()                       # обновить веса

            total_loss += loss.item() * yb.size(0)
            total_correct += (logits.argmax(dim=1) == yb).sum().item()
            total_count += yb.size(0)

        train_loss = total_loss / total_count
        train_acc = total_correct / total_count
        valid_loss, valid_acc = evaluate_torch(model, valid_loader, criterion)
        scheduler.step(valid_loss)

        history["train_loss"].append(train_loss)
        history["train_acc"].append(train_acc)
        history["valid_loss"].append(valid_loss)
        history["valid_acc"].append(valid_acc)

        if epoch == 1 or epoch % 5 == 0 or epoch == epochs:
            print(
                f"epoch {epoch:02d} | "
                f"train loss {train_loss:.4f}, acc {train_acc:.3f} | "
                f"valid loss {valid_loss:.4f}, acc {valid_acc:.3f}"
            )

    return history


torch.manual_seed(SEED)
model_torch = build_torch_mlp(
    input_dim=INPUT_DIM,
    hidden_dim=HIDDEN_DIM,
    num_classes=NUM_CLASSES,
    dropout=DROPOUT,
).to(DEVICE)
history_torch = train_torch_classifier(model_torch, train_loader, valid_loader)
plot_history_curves(history_torch, "PyTorch MLP")


In [ ]:
@torch.no_grad()
def predict_torch_proba(X_np: np.ndarray) -> np.ndarray:
    model_torch.eval()
    xb = torch.tensor(X_np, dtype=torch.float32, device=DEVICE)
    logits = model_torch(xb)
    return torch.softmax(logits, dim=1)[:, 1].cpu().numpy()


plot_binary_decision_boundary(predict_torch_proba, X, y, "PyTorch decision boundary")


### Что менять в PyTorch под свою задачу

Минимальная карта изменений:

| Что меняется в задаче | Где менять |
|---|---|
| Другое число входных признаков | INPUT_DIM и первый nn.Linear |
| Другое число классов | NUM_CLASSES и последний nn.Linear |
| Больше/меньше емкость модели | HIDDEN_DIM, число Linear/ReLU блоков |
| Переобучение | увеличить Dropout, weight_decay, добавить early stopping |
| Недообучение | увеличить HIDDEN_DIM, число слоев или epochs |
| Бинарная классификация с одним выходом | nn.Linear(..., 1) + BCEWithLogitsLoss |
| Регрессия | последний Linear с нужным числом выходов + MSELoss/SmoothL1Loss |

Ключевая мысль: в PyTorch вы сами управляете циклом обучения. Это больше кода, но студент видит весь механизм: где считаются logits, где считается loss, где появляются градиенты и где обновляются веса.


## 4. Keras / TensorFlow: быстрый высокоуровневый workflow

Keras находится внутри TensorFlow и дает более короткий pipeline для типовых задач.

Основные элементы Keras/TensorFlow pipeline:

1. tf.Tensor: структура данных TensorFlow.
2. keras.Input: описание формы одного объекта без batch dimension.
3. keras.Sequential: модель как простой стек слоев.
4. Functional API: модель как граф tensor -> layers -> output, удобен для нескольких входов/выходов и skip connections.
5. layers.Dense: полносвязный слой.
6. layers.BatchNormalization: нормализация активаций.
7. layers.Dropout: регуляризация.
8. model.compile: выбор optimizer, loss и metrics.
9. model.fit: готовый цикл обучения.
10. model.evaluate: оценка на validation/test данных.
11. model.predict: inference.
12. callbacks: дополнительные действия во время обучения, например EarlyStopping и ReduceLROnPlateau.

Pipeline работы:

1. Подготовить X_train, y_train, X_valid, y_valid как NumPy arrays или tf.data.Dataset.
2. Собрать модель через keras.Sequential или Functional API.
3. Вызвать model.compile(...).
4. Обучить через model.fit(...).
5. Посмотреть history.history и графики train/validation.
6. Оценить модель через model.evaluate(...).
7. Получить прогнозы через model.predict(...).

Keras скрывает большую часть ручного train loop. Это удобно для быстрого baseline и для студентов, которым сначала важно увидеть общий workflow.


In [ ]:
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

keras.utils.set_random_seed(SEED)
print("TensorFlow:", tf.__version__)

keras_model = keras.Sequential(
    [
        keras.Input(shape=(2,)),
        layers.Dense(64, activation="relu"),
        layers.BatchNormalization(),
        layers.Dropout(0.10),
        layers.Dense(64, activation="relu"),
        layers.Dense(2),
    ],
    name="keras_mlp",
)

keras_model.compile(
    optimizer=keras.optimizers.AdamW(learning_rate=3e-3, weight_decay=1e-4),
    loss=keras.losses.SparseCategoricalCrossentropy(from_logits=True),
    metrics=["accuracy"],
)

keras_model.summary()


In [ ]:
callbacks = [
    keras.callbacks.ReduceLROnPlateau(
        monitor="val_loss",
        factor=0.5,
        patience=4,
        verbose=1,
    ),
    keras.callbacks.EarlyStopping(
        monitor="val_loss",
        patience=8,
        restore_best_weights=True,
    ),
]

history_keras_raw = keras_model.fit(
    X_train,
    y_train,
    validation_data=(X_valid, y_valid),
    epochs=45,
    batch_size=64,
    callbacks=callbacks,
    verbose=0,
)

keras_eval = keras_model.evaluate(X_valid, y_valid, verbose=0)
print(dict(zip(keras_model.metrics_names, keras_eval)))


In [ ]:
history_keras = {
    "train_loss": history_keras_raw.history["loss"],
    "train_acc": history_keras_raw.history["accuracy"],
    "valid_loss": history_keras_raw.history["val_loss"],
    "valid_acc": history_keras_raw.history["val_accuracy"],
}
plot_history_curves(history_keras, "Keras MLP")


def predict_keras_proba(X_np: np.ndarray) -> np.ndarray:
    logits = keras_model.predict(X_np, verbose=0)
    probabilities = tf.nn.softmax(logits, axis=1).numpy()
    return probabilities[:, 1]


plot_binary_decision_boundary(predict_keras_proba, X, y, "Keras decision boundary")


### Functional API в Keras

Sequential подходит для линейного стека слоев. Если нужны несколько входов, несколько выходов, skip connections или shared layers, используйте Functional API.


In [ ]:
inputs = keras.Input(shape=(2,), name="features")
x = layers.Dense(64, activation="relu")(inputs)
x = layers.Dropout(0.10)(x)
x = layers.Dense(64, activation="relu")(x)
outputs = layers.Dense(2, name="logits")(x)

functional_model = keras.Model(inputs=inputs, outputs=outputs, name="functional_mlp")
functional_model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=3e-3),
    loss=keras.losses.SparseCategoricalCrossentropy(from_logits=True),
    metrics=["accuracy"],
)
functional_model.summary()


### Что менять в Keras под свою задачу

Минимальная карта изменений:

| Что меняется в задаче | Где менять |
|---|---|
| Другая форма входа | keras.Input(shape=...) |
| Другое число классов | последний layers.Dense(num_classes) |
| Бинарная классификация | layers.Dense(1) + BinaryCrossentropy(from_logits=True) |
| Multiclass с целыми метками | SparseCategoricalCrossentropy(from_logits=True) |
| Multiclass с one-hot метками | CategoricalCrossentropy(from_logits=True) |
| Регрессия | Dense(n_outputs) + MSE/MAE/Huber loss |
| Переобучение | Dropout, weight_decay, EarlyStopping, больше данных/augmentation |
| Недообучение | больше слоев/нейронов, больше epochs, другой learning rate |

Особенно важное правило: если loss указан с `from_logits=True`, последний слой должен возвращать сырые logits, то есть без softmax/sigmoid. Вероятности можно получить отдельно на inference.


## 5. Hugging Face: модели Transformers

Hugging Face Transformers обычно используют не для MLP с нуля, а для pretrained-моделей и Transformer-архитектур.

Основные элементы Hugging Face pipeline:

1. Tokenizer: превращает текст в input_ids, attention_mask и иногда token_type_ids.
2. Config: описание архитектуры модели: размер словаря, число слоев, hidden size, число labels.
3. Model: готовая архитектура, например AutoModelForSequenceClassification или DistilBertForSequenceClassification.
4. Dataset format: каждый пример обычно является словарем с ключами input_ids, attention_mask, labels.
5. Data collator: собирает примеры в batch, часто также делает padding.
6. TrainingArguments: гиперпараметры и служебные настройки обучения.
7. Trainer: готовый training/evaluation loop для Transformers.
8. compute_metrics: функция для метрик на validation.
9. pipeline: простой inference API поверх tokenizer + model.
10. Hub: хранилище моделей, tokenizer-ов и датасетов.

Pipeline работы для реального текста:

1. Подготовить texts и labels.
2. Выбрать model_name, например русскоязычный или multilingual Transformer.
3. Загрузить AutoTokenizer.from_pretrained(model_name).
4. Токенизировать тексты: tokenizer(texts, truncation=True, padding=True, max_length=...).
5. Собрать dataset как список словарей или datasets.Dataset.
6. Загрузить AutoModelForSequenceClassification.from_pretrained(..., num_labels=...).
7. Задать TrainingArguments.
8. Создать Trainer и вызвать trainer.train().
9. Проверить trainer.evaluate().
10. Использовать pipeline или ручной tokenizer + model для inference.

В первом примере ниже pretrained-веса не скачиваются: маленький DistilBERT создается из config со случайными весами. Это учебный пример API, а не способ получить хорошее качество на реальной NLP-задаче.


In [ ]:
try:
    import inspect
    import torch
    from transformers import (
        DistilBertConfig,
        DistilBertForSequenceClassification,
        Trainer,
        TrainingArguments,
        set_seed as hf_set_seed,
    )
except ModuleNotFoundError as exc:
    raise ModuleNotFoundError(
        "Для Hugging Face-раздела нужны torch, transformers и accelerate. "
        "Установите: python -m pip install torch transformers accelerate"
    ) from exc

hf_set_seed(SEED)
print("Hugging Face Transformers импортирован")


In [ ]:
def make_rule_token_dataset(n: int = 256, seq_len: int = 16, vocab_size: int = 64, seed: int = 42):
    """Toy dataset без собственного класса.

    Каждый пример — обычный dict, как ожидает Hugging Face Trainer:
    input_ids, attention_mask, labels.
    label=1, если сумма первой половины токенов больше суммы второй половины.
    """
    rng = np.random.default_rng(seed)
    input_ids_np = rng.integers(5, vocab_size, size=(n, seq_len), dtype=np.int64)
    attention_mask_np = np.ones_like(input_ids_np, dtype=np.int64)
    labels_np = (input_ids_np[:, : seq_len // 2].sum(axis=1) > input_ids_np[:, seq_len // 2 :].sum(axis=1)).astype(np.int64)

    input_ids = torch.tensor(input_ids_np, dtype=torch.long)
    attention_mask = torch.tensor(attention_mask_np, dtype=torch.long)
    labels = torch.tensor(labels_np, dtype=torch.long)

    dataset = []
    for i in range(n):
        dataset.append(
            {
                "input_ids": input_ids[i],
                "attention_mask": attention_mask[i],
                "labels": labels[i],
            }
        )

    return dataset, labels


hf_train_dataset, hf_train_labels = make_rule_token_dataset(n=320, seed=SEED)
hf_valid_dataset, hf_valid_labels = make_rule_token_dataset(n=96, seed=SEED + 1)

example = hf_train_dataset[0]
print(example)
print("train labels:", torch.bincount(hf_train_labels))


In [ ]:
config = DistilBertConfig(
    vocab_size=64,
    max_position_embeddings=32,
    n_layers=2,
    dim=64,
    hidden_dim=128,
    n_heads=2,
    num_labels=2,
    id2label={0: "RIGHT_SUM_BIGGER", 1: "LEFT_SUM_BIGGER"},
    label2id={"RIGHT_SUM_BIGGER": 0, "LEFT_SUM_BIGGER": 1},
)

hf_model = DistilBertForSequenceClassification(config)
print("parameters:", sum(p.numel() for p in hf_model.parameters()))

with torch.no_grad():
    batch = {key: value.unsqueeze(0) for key, value in example.items()}
    output = hf_model(**batch)
print("loss:", float(output.loss))
print("logits:", output.logits)


In [ ]:
def make_training_args(**kwargs) -> TrainingArguments:
    """Совместимость с новыми и старыми версиями transformers: eval_strategy/evaluation_strategy."""
    parameters = inspect.signature(TrainingArguments.__init__).parameters
    if "eval_strategy" not in parameters and "eval_strategy" in kwargs:
        kwargs["evaluation_strategy"] = kwargs.pop("eval_strategy")
    return TrainingArguments(**kwargs)


def compute_accuracy(eval_pred) -> Dict[str, float]:
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    return {"accuracy": float((predictions == labels).mean())}


training_args = make_training_args(
    output_dir="./hf_toy_runs",
    learning_rate=5e-4,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    num_train_epochs=3,
    weight_decay=0.01,
    eval_strategy="epoch",
    logging_strategy="epoch",
    save_strategy="no",
    report_to="none",
    disable_tqdm=True,
)

trainer = Trainer(
    model=hf_model,
    args=training_args,
    train_dataset=hf_train_dataset,
    eval_dataset=hf_valid_dataset,
    compute_metrics=compute_accuracy,
)

trainer.train()
trainer.evaluate()


### Fine-tuning pretrained-модели Hugging Face

Следующая ячейка является шаблоном для реального fine-tuning текста. По умолчанию она выключена, потому что при первом запуске скачивает tokenizer и модель.

Что происходит в pipeline:

1. `texts` и `labels` — ваши обучающие данные.
2. `AutoTokenizer` превращает строки в числовые токены.
3. `encoded` содержит `input_ids` и `attention_mask`.
4. `dataset` собирается как список словарей без собственного класса.
5. `AutoModelForSequenceClassification` загружает pretrained Transformer и добавляет классификационную голову.
6. `TrainingArguments` задает learning rate, batch size, число эпох и сохранение.
7. `Trainer` запускает обучение.

Что нужно заменить под свою задачу:

- texts и labels заменить на ваши данные.
- num_labels, id2label, label2id заменить под ваши классы.
- model_name выбрать под язык и размер модели.
- max_length подобрать по длине текстов.
- learning_rate обычно начинать с 1e-5 ... 5e-5 для pretrained Transformers.


In [ ]:
RUN_PRETRAINED_FINE_TUNING = False

if RUN_PRETRAINED_FINE_TUNING:
    from transformers import AutoModelForSequenceClassification, AutoTokenizer

    model_name = "distilbert/distilbert-base-uncased"
    texts = [
        "The course was clear and practical.",
        "The explanation was confusing and too fast.",
        "I like this lesson.",
        "This example is not useful.",
        "The model works well on my task.",
        "The result is bad and unstable.",
    ]
    labels = [1, 0, 1, 0, 1, 0]

    tokenizer = AutoTokenizer.from_pretrained(model_name)
    encoded = tokenizer(
        texts,
        truncation=True,
        padding=True,
        max_length=64,
        return_tensors="pt",
    )

    dataset = []
    for i in range(len(labels)):
        item = {key: value[i] for key, value in encoded.items()}
        item["labels"] = torch.tensor(labels[i], dtype=torch.long)
        dataset.append(item)

    id2label = {0: "NEGATIVE", 1: "POSITIVE"}
    label2id = {value: key for key, value in id2label.items()}
    pretrained_model = AutoModelForSequenceClassification.from_pretrained(
        model_name,
        num_labels=2,
        id2label=id2label,
        label2id=label2id,
    )

    args = make_training_args(
        output_dir="./hf_text_runs",
        learning_rate=2e-5,
        per_device_train_batch_size=4,
        num_train_epochs=3,
        weight_decay=0.01,
        save_strategy="no",
        report_to="none",
    )

    text_trainer = Trainer(
        model=pretrained_model,
        args=args,
        train_dataset=dataset,
    )
    text_trainer.train()


### Inference через pipeline

pipeline удобен для быстрого применения уже обученной модели. Эта ячейка тоже выключена по умолчанию, потому что может скачивать модель.


In [ ]:
RUN_PIPELINE_INFERENCE = False

if RUN_PIPELINE_INFERENCE:
    from transformers import pipeline

    classifier = pipeline(
        "sentiment-analysis",
        model="distilbert/distilbert-base-uncased-finetuned-sst-2-english",
    )
    print(classifier("This lesson gives practical building blocks."))


## 6. Сравнение трех подходов

| Подход | Сильные стороны | Когда выбирать | Ментальная модель pipeline |
|---|---|---|---|
| PyTorch | Максимальный контроль, прозрачный train loop, гибкость | Исследования, нестандартное обучение, production ML с кастомной логикой | Вы сами пишете цикл: batch -> forward -> loss -> backward -> step |
| Keras/TensorFlow | Короткий код, быстрый baseline, удобные callbacks | Быстро собрать и проверить модель, учебные проекты, стандартные pipeline | Вы описываете модель и параметры обучения, model.fit делает цикл |
| Hugging Face | Pretrained Transformers, tokenizer/model ecosystem, Trainer, Hub | NLP, multimodal Transformers, fine-tuning готовых моделей | tokenizer готовит входы, Trainer обучает model на словарях input_ids/labels |

Практический принцип: сначала сделайте простой baseline, затем усложняйте архитектуру и pipeline только когда понятно, что именно ограничивает качество.
